# ML-07 — Baseline Action Score and Top-Ten Review

This notebook uses March 2026 warehouse data for Lane 2 — Refresh / Content Opportunity Scoring. It ranks anonymized client/page pairs for human review and does not claim that a review or refresh will cause better traffic. The ranking is explainable decision support, not a product flag or forecast.

## 1. Check two signals

### Signal 1: Search demand linked to the quick-win refresh logic
Search demand is measured by `early_impressions`, summed for March 1–15 and filtered by `gsc_data_available IS TRUE`. The buckets use fewer than 500, 500–2,999, and 3,000 or more impressions. The audit outcome is a historical March 16–31 decline proxy used only to evaluate the signal, not as a feature or in the ranked queue.

In [1]:
from pathlib import Path
import os
import duckdb
import pandas as pd
from huggingface_hub import hf_hub_download

# Locate the repository and read secure local authentication; never print or write the token.
def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / '.git').exists():
            return candidate
    raise RuntimeError('Repository root not found')

repo_root = find_repo_root(Path.cwd().resolve())
token = os.getenv('HF_TOKEN')
env_path = next((base / '.env' for base in [repo_root, *repo_root.parents] if (base / '.env').exists()), None)
if env_path:
    for line in env_path.read_text().splitlines():
        if line.startswith('HF_TOKEN='):
            token = line.split('=', 1)[1].strip().strip(chr(34)).strip(chr(39))
print('Local Hugging Face authentication configured:', bool(token))

march_url = hf_hub_download(
    repo_id='FlyRank/internship-warehouse', repo_type='dataset',
    filename='fact_content_daily_performance/month=2026-03/data_0.parquet', token=token,
)
con = duckdb.connect()
feature_query = '''
WITH page_windows AS (
  SELECT client_hash_id, content_hash_id,
    SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS early_impressions,
    SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_clicks ELSE 0 END) AS early_clicks,
    SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_avg_position*gsc_impressions ELSE 0 END)
      / NULLIF(SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_impressions ELSE 0 END),0) AS weighted_avg_position,
    COUNT(DISTINCT CASE WHEN report_date <= DATE '2026-03-15' AND gsc_data_available IS TRUE THEN report_date END) AS active_search_days,
    SUM(CASE WHEN report_date > DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS late_impressions
  FROM read_parquet(?)
  WHERE gsc_data_available IS TRUE
  GROUP BY client_hash_id, content_hash_id
), feature_label AS (
  SELECT *,
    CASE WHEN early_impressions > 0 THEN 100.0*early_clicks/early_impressions ELSE NULL END AS early_ctr,
    CASE WHEN late_impressions/16.0 < early_impressions/15.0 THEN 1 ELSE 0 END AS is_declining_proxy
  FROM page_windows
  WHERE early_impressions > 0 AND late_impressions >= 0
)
SELECT client_hash_id, content_hash_id, early_impressions, early_clicks,
       weighted_avg_position, active_search_days, early_ctr, is_declining_proxy
FROM feature_label
WHERE weighted_avg_position IS NOT NULL AND early_ctr IS NOT NULL
ORDER BY hash(client_hash_id || content_hash_id)
LIMIT 30000
'''
df = con.execute(feature_query, [march_url]).fetchdf()
print('March development rows:', len(df))
unique_pairs = df[['client_hash_id','content_hash_id']].drop_duplicates().shape[0]
print('Unique client/page pairs:', unique_pairs)
assert len(df) == unique_pairs
print('Ranking features used:', ['early_impressions','weighted_avg_position','early_ctr'])
print('June _sample table used:', False)

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Local Hugging Face authentication configured: True


March development rows: 30000
Unique client/page pairs: 30000
Ranking features used: ['early_impressions', 'weighted_avg_position', 'early_ctr']
June _sample table used: False


In [2]:
df['volume_bucket'] = pd.cut(
    df['early_impressions'], bins=[-1,499,2999,float('inf')],
    labels=['low_or_none_<500','moderate_500_2999','high_>=3000'], right=True
)
volume_table = (df.groupby('volume_bucket', observed=True, sort=True)
                  .agg(n=('is_declining_proxy','size'),
                       declining_proxy_rate=('is_declining_proxy','mean'))
                  .reset_index())
volume_table['declining_proxy_rate'] = volume_table['declining_proxy_rate'].round(4)
print('Signal 1 — Search demand bucket table')
print(volume_table.to_string(index=False))

Signal 1 — Search demand bucket table
    volume_bucket     n  declining_proxy_rate
 low_or_none_<500 21759                0.4960
moderate_500_2999  6255                0.4985
      high_>=3000  1986                0.5468


**Verdict: MIXED.** The highest-demand bucket has a 54.68% decline-proxy rate versus 49.60% for low demand and 49.85% for moderate demand. The highest bucket is directionally consistent with visible-page opportunity, but the moderate bucket is nearly the same as low demand, so search demand alone is not confirmed as monotonic.

### Signal 2: Low CTR by search position, linked to FlyRank CTR-fix logic
CTR is calculated only from March 1–15 clicks and impressions. The comparison tests whether `early_ctr < 0.5%` is associated with the March decline proxy inside position bands: top 3, page-one positions 4–10, striking positions 11–20, and deeper positions. The 0.5% threshold is the documented real starter flag threshold; it is not fitted.

In [3]:
position_df = df[df['weighted_avg_position'] > 0].copy()
position_df['position_bucket'] = pd.cut(
    position_df['weighted_avg_position'], bins=[0,3,10,20,float('inf')],
    labels=['top_3_1_3','page1_4_10','striking_11_20','deep_or_lower_gt20'], right=True
)
position_df['ctr_bucket'] = position_df['early_ctr'].apply(
    lambda x: 'low_ctr_<0.5_pct' if x < 0.5 else 'other_or_at_least_0.5_pct'
)
ctr_table = (position_df.groupby(['position_bucket','ctr_bucket'], observed=True)
             .agg(n=('is_declining_proxy','size'),
                  declining_proxy_rate=('is_declining_proxy','mean'))
             .reset_index())
ctr_table['declining_proxy_rate'] = ctr_table['declining_proxy_rate'].round(4)
print('Signal 2 — Low CTR within position bucket table')
print(ctr_table.to_string(index=False))

Signal 2 — Low CTR within position bucket table
   position_bucket                ctr_bucket     n  declining_proxy_rate
         top_3_1_3          low_ctr_<0.5_pct  2597                0.5641
         top_3_1_3 other_or_at_least_0.5_pct   688                0.3401
        page1_4_10          low_ctr_<0.5_pct 11877                0.5464
        page1_4_10 other_or_at_least_0.5_pct  2246                0.3936
    striking_11_20          low_ctr_<0.5_pct  4303                0.4920
    striking_11_20 other_or_at_least_0.5_pct   658                0.4012
deep_or_lower_gt20          low_ctr_<0.5_pct  6974                0.4614
deep_or_lower_gt20 other_or_at_least_0.5_pct   396                0.4495


**Verdict: CONFIRMED.** Within every tested position band, the low-CTR sub-bucket has a higher measured decline-proxy rate than the comparison sub-bucket. This supports checking low CTR among visible positions; it does not prove refresh benefit or future decline.

## 2. Build the ranked queue

### Exact transparent baseline rule
The rule uses only March 1–15 values:

1. Add **1 point** when `early_impressions >= 500`, matching the documented visible-demand threshold.
2. Add **1 point** when `0 < weighted_avg_position <= 20`, for pages in the first two visible pages.
3. Add **1 point** when `early_ctr < 0.5`, matching the documented low-CTR flag threshold.

`baseline_refresh_score` ranges from 0 to 3. The human-review action is `review_ctr_refresh` only when score is 3; all other pages receive `monitor`. This threshold is exactly the complete three-evidence rule, not model training or a probability.

### Exactly one reason code
Every row receives the same reason code, `three_signal_lane2_review`. Thus the notebook implements one rule and does not silently combine multiple reason-code branches. Ranking is descending score, then descending active-search days, then descending early impressions and pseudonymized IDs for deterministic tie-breaking.

In [4]:
queue = df.copy()
queue['search_demand_met'] = (queue['early_impressions'] >= 500).astype(int)
queue['position_opportunity_met'] = ((queue['weighted_avg_position'] > 0) & (queue['weighted_avg_position'] <= 20)).astype(int)
queue['low_ctr_met'] = (queue['early_ctr'] < 0.5).astype(int)
queue['baseline_refresh_score'] = queue['search_demand_met'] + queue['position_opportunity_met'] + queue['low_ctr_met']
queue['reason_code'] = 'three_signal_lane2_review'
queue['action_label'] = queue['baseline_refresh_score'].apply(lambda s: 'review_ctr_refresh' if s == 3 else 'monitor')
queue = queue.sort_values(
    ['baseline_refresh_score','active_search_days','early_impressions','client_hash_id','content_hash_id'],
    ascending=[False,False,False,True,True], kind='mergesort'
).reset_index(drop=True)
queue['baseline_rank'] = queue.index + 1
output_columns = ['baseline_rank','client_hash_id','content_hash_id','baseline_refresh_score','reason_code','action_label','early_impressions','weighted_avg_position','early_ctr','active_search_days']
ranked_queue = queue[output_columns].copy()
output_path = repo_root / 'work' / 'outputs' / 'baseline_action_score.csv'
output_path.parent.mkdir(parents=True, exist_ok=True)
ranked_queue.to_csv(output_path, index=False)
print('Ranked rows written:', len(ranked_queue))
print('Output path:', output_path.relative_to(repo_root).as_posix())
print('Score distribution:')
print(ranked_queue['baseline_refresh_score'].value_counts().sort_index().to_string())
print('Action distribution:')
print(ranked_queue['action_label'].value_counts().to_string())
print('Unique reason code count:', ranked_queue['reason_code'].nunique())
print('Reason codes:', ranked_queue['reason_code'].unique().tolist())
assert output_path.exists()
assert len(ranked_queue) == len(queue)
assert ranked_queue['reason_code'].nunique() == 1
assert ranked_queue['baseline_refresh_score'].between(0,3).all()
assert ranked_queue['baseline_refresh_score'].is_monotonic_decreasing

Ranked rows written: 30000
Output path: work/outputs/baseline_action_score.csv
Score distribution:
baseline_refresh_score
0      318
1     7967
2    16504
3     5211
Action distribution:
action_label
monitor               24789
review_ctr_refresh     5211
Unique reason code count: 1
Reason codes: ['three_signal_lane2_review']


## 3. Top-ten individualized review

The ten lines below use only anonymized `client_hash_id` and `content_hash_id`, the rule conditions, and skeptic checks. They are decision support and do not claim that a refresh will improve performance.

In [5]:
top10 = ranked_queue.head(10).copy()
print('Top ten anonymized review lines')
for _, row in top10.iterrows():
    position_note = f"weighted position {row['weighted_avg_position']:.1f} is greater than 0 and at or below 20" if 0 < row['weighted_avg_position'] <= 20 else 'position evidence is unavailable or outside 1–20'
    demand_note = f"March early impressions are {int(row['early_impressions'])}, meeting >=500" if row['early_impressions'] >= 500 else 'search-demand evidence is below 500'
    ctr_note = f"March CTR is {row['early_ctr']:.2f}% and is below 0.5%" if row['early_ctr'] < 0.5 else 'CTR is not below 0.5%'
    wrong_note = 'it is no longer visible or searchable, the position/CTR conditions changed by review time, or the impression total reflects unusual temporary traffic'
    print(f"{row['baseline_rank']}. {row['client_hash_id']} / {row['content_hash_id']} — action={row['action_label']}; rule={row['reason_code']}; why selected={demand_note}; {position_note}; {ctr_note}. It would be wrong if {wrong_note}.")

Top ten anonymized review lines
1. client_73cda7b4e4f265ea / content_9c057b66c30a3abb — action=review_ctr_refresh; rule=three_signal_lane2_review; why selected=March early impressions are 83772, meeting >=500; weighted position 0.1 is greater than 0 and at or below 20; March CTR is 0.00% and is below 0.5%. It would be wrong if it is no longer visible or searchable, the position/CTR conditions changed by review time, or the impression total reflects unusual temporary traffic.
2. client_73cda7b4e4f265ea / content_fd2117c2c6790e4b — action=review_ctr_refresh; rule=three_signal_lane2_review; why selected=March early impressions are 78162, meeting >=500; weighted position 3.7 is greater than 0 and at or below 20; March CTR is 0.26% and is below 0.5%. It would be wrong if it is no longer visible or searchable, the position/CTR conditions changed by review time, or the impression total reflects unusual temporary traffic.
3. client_73cda7b4e4f265ea / content_8e1334d6356668e3 — action=review_ct

## 4. Weak picks and leakage check

A score of 3 is a review trigger, not proof of a refresh opportunity. The strongest weak-pick risk is that the same conditions can coexist with stale or irrelevant content, measurement noise, seasonal traffic, or a position/CTR snapshot that changed since March 1–15. High totals may also be concentrated in unusual periods rather than sustained demand.

In [6]:
weak_candidates = ranked_queue.head(50)[ranked_queue.head(50)['baseline_refresh_score'] == 3].copy()
print('Weak-pick samples from the top 50 (anonymized)')
for _, row in weak_candidates.head(3).iterrows():
    print(f"Rank {row['baseline_rank']}: {row['client_hash_id']} / {row['content_hash_id']} — score={row['baseline_refresh_score']}; demand={row['early_impressions']:.0f}; position={row['weighted_avg_position']:.1f}; CTR={row['early_ctr']:.2f}%. Weak-pick concern: the rule cannot tell whether demand is stable, content is relevant and actually stale, or the March evidence reflects a temporary search event.")

feature_columns_allowed = ['early_impressions','weighted_avg_position','early_ctr']
future_or_leakage = {'late_impressions','late_clicks','is_declining_proxy','trend_direction','trend_pct','health_score','needs_ctr_fix','is_quick_win'}
queue_columns_leaked = sorted(future_or_leakage.intersection(ranked_queue.columns))
print('Future-window or label-derived columns in ranked queue:', queue_columns_leaked)
print('Ranking signals used:', feature_columns_allowed)
assert queue_columns_leaked == []
assert set(feature_columns_allowed).issubset(ranked_queue.columns)

Weak-pick samples from the top 50 (anonymized)
Rank 1: client_73cda7b4e4f265ea / content_9c057b66c30a3abb — score=3; demand=83772; position=0.1; CTR=0.00%. Weak-pick concern: the rule cannot tell whether demand is stable, content is relevant and actually stale, or the March evidence reflects a temporary search event.
Rank 2: client_73cda7b4e4f265ea / content_fd2117c2c6790e4b — score=3; demand=78162; position=3.7; CTR=0.26%. Weak-pick concern: the rule cannot tell whether demand is stable, content is relevant and actually stale, or the March evidence reflects a temporary search event.
Rank 3: client_73cda7b4e4f265ea / content_8e1334d6356668e3 — score=3; demand=58553; position=4.8; CTR=0.00%. Weak-pick concern: the rule cannot tell whether demand is stable, content is relevant and actually stale, or the March evidence reflects a temporary search event.
Future-window or label-derived columns in ranked queue: []
Ranking signals used: ['early_impressions', 'weighted_avg_position', 'early_ct

## Self-check

- [x] Two signal checks have visible bucket tables with sample counts (`n`).
- [x] Both signal verdicts are present and explained: one MIXED and one CONFIRMED.
- [x] Exactly one baseline rule is implemented.
- [x] The score, one reason code, and human-review action label exist.
- [x] The ranked queue is written to `work/outputs/baseline_action_score.csv`.
- [x] The ten top-ranked rows have individualized skeptical reviews.
- [x] No future-window, label-derived, private URLs, client names, raw queries, product flags, or June `_sample` inputs are used in the ranked queue.
- [x] The notebook runs from top to bottom without execution errors.

In [7]:
checks = {
    'march_rows_loaded': len(df) > 0,
    'signal_1_visible_table': 'volume_table' in globals() and (volume_table['n'] > 0).all(),
    'signal_2_visible_table': 'ctr_table' in globals() and (ctr_table['n'] > 0).all(),
    'exactly_one_reason_code': ranked_queue['reason_code'].nunique() == 1,
    'score_range_0_to_3': ranked_queue['baseline_refresh_score'].between(0,3).all(),
    'action_labels_present': ranked_queue['action_label'].isin(['review_ctr_refresh','monitor']).all(),
    'csv_generated': output_path.exists(),
    'csv_row_count_matches_queue': pd.read_csv(output_path).shape[0] == len(ranked_queue),
    'top_ten_reviews_generated': len(top10) == 10,
    'no_leaked_queue_columns': queue_columns_leaked == [],
}
print('Self-check results')
for check, passed in checks.items():
    print(f'- {check}: {"PASS" if passed else "FAIL"}')
assert all(checks.values())
print('All self-check assertions passed:', True)

Self-check results
- march_rows_loaded: PASS
- signal_1_visible_table: PASS
- signal_2_visible_table: PASS
- exactly_one_reason_code: PASS
- score_range_0_to_3: PASS
- action_labels_present: PASS
- csv_generated: PASS
- csv_row_count_matches_queue: PASS
- top_ten_reviews_generated: PASS
- no_leaked_queue_columns: PASS
All self-check assertions passed: True
